<a href="https://colab.research.google.com/github/KoraRiko/joke_generator/blob/main/AI_Joke_Metrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installing libraries
!pip install openai sentence-transformers transformers torch scikit-learn nltk sacrebleu pandas tqdm
import nltk
nltk.download('punkt')

In [38]:
import os
import openai
import pandas as pd
from tqdm import tqdm
from itertools import combinations
import numpy as np

# setting OpenAI
from google.colab import userdata
#Runtime -> "User-provided secrets" -> secret OPENAI_API_KEY
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
KEYWORDS = ["girlfriend", "birth", "letter", "princess", "sheep", "board", "hipsters"]

NUM_SAMPLES_PER_KEYWORD = 10  # How many jokes are generated per word

Fine tuned

In [ ]:
from openai import OpenAI

def generate_joke(keyword, model="ft:gpt-3.5-turbo-0125:korariko::DQ1ft67K", temperature=0.5):

    SYSTEM_PROMPT = """You are a professional comedy writer specializing in clever wordplay and safe-but-edgy humor. Apply the Benign Violation principle — break an expectation or norm, but keep it safe and clever."""

    USER_PROMPT = f"Generate a short, clever joke about: {keyword}. Joke must visible include the word '{keyword}'."
    try:
        client = OpenAI()  # automatically reads OPENAI_API_KEY from environment
        response = client.chat.completions.create(
            model=model,
            messages=[
              {"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": USER_PROMPT},
            ],
            temperature=temperature,
            max_tokens=80
        )
        joke = response.choices[0].message.content.strip()  # .content, not ['content']
        return joke
    except Exception as e:
        print(f"Ошибка при генерации для '{keyword}': {e}")
        return None
#Генирация
all_jokes = []
for keyword in tqdm(KEYWORDS, desc="Генерация шуток"):
    for i in range(NUM_SAMPLES_PER_KEYWORD):
        joke = generate_joke(keyword)
        if joke:
            all_jokes.append({"keyword": keyword, "joke": joke})
        else:
            all_jokes.append({"keyword": keyword, "joke": "[ERROR]"})

pd.set_option('display.max_colwidth', None)
df = pd.DataFrame(all_jokes)
print("\nСгенерировано шуток:", len(df))
display(df)

without fine tune

In [ ]:
from openai import OpenAI

def generate_joke(keyword, model="gpt-3.5-turbo", temperature=0.5):

    SYSTEM_PROMPT = """You are a professional comedy writer specializing in clever wordplay and safe-but-edgy humor. Apply the Benign Violation principle — break an expectation or norm, but keep it safe and clever."""

    USER_PROMPT = f"Generate a short, clever joke about: {keyword}. Joke must visible include the word '{keyword}'."
    try:
        client = OpenAI()  # automatically reads OPENAI_API_KEY from environment
        response = client.chat.completions.create(
            model=model,
            messages=[
              {"role": "system", "content": SYSTEM_PROMPT},
              {"role": "user", "content": USER_PROMPT},
            ],
            temperature=temperature,
            max_tokens=80
        )
        joke = response.choices[0].message.content.strip()  # .content, not ['content']
        return joke
    except Exception as e:
        print(f"Ошибка при генерации для '{keyword}': {e}")
        return None
#Generation
all_jokes = []
for keyword in tqdm(KEYWORDS, desc="Генерация шуток"):
    for i in range(NUM_SAMPLES_PER_KEYWORD):
        joke = generate_joke(keyword)
        if joke:
            all_jokes.append({"keyword": keyword, "joke": joke})
        else:
            all_jokes.append({"keyword": keyword, "joke": "[ERROR]"})

pd.set_option('display.max_colwidth', None)
df = pd.DataFrame(all_jokes)
print("\nСгенерировано шуток:", len(df))
display(df)

In [ ]:
!pip install sacrebleu

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline, GPT2LMHeadModel, GPT2Tokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sacrebleu.metrics import BLEU
import torch

# Semantic model
sem_model = SentenceTransformer('all-MiniLM-L6-v2')

# Humor classifier (public model for humour)
humor_pipe = pipeline(
    "text-classification",
    model="likhithasapu/humour-detection-mBert",
    truncation=True,
    max_length=128
)

# Perplexity (GPT-2)
ppl_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
ppl_model = GPT2LMHeadModel.from_pretrained("gpt2")
ppl_model.eval()

# BLEU scorer
bleu_scorer = BLEU(effective_order=True)

In [42]:
def compute_semantic_distance(keyword, joke):
    emb_kw = sem_model.encode([keyword])
    emb_joke = sem_model.encode([joke])
    sim = cosine_similarity(emb_kw, emb_joke)[0][0]
    return float(1 - sim)

def compute_perplexity(text):
    inputs = ppl_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = ppl_model(**inputs, labels=inputs["input_ids"])
        return float(torch.exp(outputs.loss).item())

def compute_humor_score(joke):
    try:
        result = humor_pipe(joke)[0]
        if result['label'] == 'HUMOR':
            return float(result['score'])
        else:
            return float(1 - result['score'])
    except:
        return 0.0

def compute_self_bleu(jokes_list):
    if len(jokes_list) < 2:
        return 0.0
    scores = []
    for i, candidate in enumerate(jokes_list):
        references = [jokes_list[j] for j in range(len(jokes_list)) if j != i]
        bleu = bleu_scorer.sentence_score(candidate, references)
        scores.append(bleu.score / 100.0)
    return np.mean(scores)

In [ ]:
from tqdm import tqdm
tqdm.pandas()

print("\n📊 Оценка метрик...")

# We only keep the best jokes
valid_df = df[df['joke'] != "[ERROR]"].copy()

# Let’s look at the metrics
valid_df['semantic_distance'] = valid_df.progress_apply(
    lambda row: compute_semantic_distance(row['keyword'], row['joke']), axis=1
)
valid_df['perplexity'] = valid_df['joke'].progress_apply(compute_perplexity)
valid_df['humor_score'] = valid_df['joke'].progress_apply(compute_humor_score)

#!!!!!!!!!!!!!!!!!!!! Novelty (Self-BLEU for all jokes)
all_valid_jokes = valid_df['joke'].tolist()
self_bleu = compute_self_bleu(all_valid_jokes)
valid_df['novelty_score'] = 1 - self_bleu

# Composite score (нормализуем и усредняем)
for col in ['semantic_distance', 'perplexity', 'humor_score']:
    valid_df[col] = pd.to_numeric(valid_df[col], errors='coerce')
    valid_df[col] = valid_df[col].fillna(valid_df[col].median())
    valid_df[f"{col}_norm"] = (valid_df[col] - valid_df[col].min()) / (valid_df[col].max() - valid_df[col].min() + 1e-8)

# Invert perplexity: high surprisal = good
valid_df['perplexity_norm'] = 1 - valid_df['perplexity_norm']

valid_df['composite_score'] = valid_df[['semantic_distance_norm', 'perplexity_norm', 'humor_score_norm']].mean(axis=1)

# Объединяем обратно с исходным df (чтобы сохранить порядок)
df = df.merge(valid_df[['keyword', 'joke', 'semantic_distance', 'perplexity', 'humor_score', 'novelty_score', 'composite_score']],
              on=['keyword', 'joke'], how='left')


In [ ]:
df.to_csv("anegen_evaluation.csv", index=False)
print("\n✅ Результаты сохранены в anegen_evaluation.csv")
print("\nТоп-5 шуток по composite_score:")
pd.set_option('display.max_colwidth', None)
display(df.nlargest(5, 'composite_score')[['keyword', 'joke', 'composite_score']])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='semantic_distance', y='humor_score', hue='keyword', s=100)
plt.title('Semantic Distance vs Humor Classifier Score')
plt.xlabel('Semantic Distance (Keyword → Joke)')
plt.ylabel('Humor Score')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['composite_score'].dropna(), bins=10, kde=True)
plt.title('Distribution of Composite Quality Score')
plt.xlabel('Composite Score')
plt.ylabel('Frequency')
plt.xlim(0, 1)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='keyword', y='composite_score')
plt.xticks(rotation=30)
plt.title('Quality of Jokes by Keyword')
plt.ylabel('Composite Score')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[['semantic_distance', 'perplexity', 'humor_score', 'composite_score']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Between Metrics')
plt.show()

In [ ]:
top5 = df.nlargest(5, 'composite_score')
plt.figure(figsize=(10, 5))
bars = plt.barh(range(len(top5)), top5['composite_score'], color='skyblue')
plt.yticks(range(len(top5)), [f"{row.keyword}: {row.joke[:100]}..." for _, row in top5.iterrows()])
plt.xlabel('Composite Score')
plt.title('Top-5 Generated Jokes by Quality')
plt.xlim(0, 1)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()